### **Tools & Binding**

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
model = ChatGoogleGenerativeAI(
    model = "gemini-3.6-flash"
)


print(f"[INFO] Model : {model.model} Loaded Successfully!")

[INFO] Model : gemini-3.6-flash Loaded Successfully!


### **DuckDuckGo Search tool**

In [2]:
from langchain_community.tools import DuckDuckGoSearchRun

duck_search = DuckDuckGoSearchRun()

duck_search.invoke("Latest Scorecard update on Ind vs Sl test series")

/tmp/ipykernel_14168/1408362800.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


'Follow SL 259/7 (73) vs IND 462 (Sonal Dinusha 90(154) Prabath Jayasuriya 5(27)) | Sri Lanka vs India, 1st Test, India tour of Sri Lanka 2026 live with live scores, ball-by-ball commentary ... Follow IND 357/6 (45) d & 214/4 vs SLCXI 363/8 d & 200/6 d (Mohammed Siraj 32 (15) Saransh Jain 2 (7)) | Sri Lanka Cricket XI vs India, 3-Day Warm-up Match, India tour of Sri Lanka 2026 live with ... Follow our live cricket update for in-depth match coverage and exciting highlights from Sri Lanka vs India 1st Test at Galle on Cricinfo. Get cricket scorecard of 1st Test, SL vs IND, India in Sri Lanka 2026 at Galle International Stadium dated August 15 - 19, 2026. Devdutt Padikkal hit a career-best 167 as India reached 460/9 against Sri Lanka on day two of the first Test, despite rain delays and regular wickets.Devdutt Padikkal 167, India vs Sri Lanka first Test, Galle Test, Padikkal career-best score, Padikkal maiden Test 150, India 460 for 9, Malayali batter Devdutt Padikkal, Sri Lanka vs India 

### **Arxiv tool - research paper**

In [3]:
# from langchain_community.tools import ArxivQueryRun
# from langchain_community.utilities import ArxivAPIWrapper

# arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper(), max_results = 2)
# arxiv_query.invoke("Transformer models in NLP")

### **Wikipedia Search Tool**

In [4]:
import wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia.set_user_agent(
    "LangGraphLearning/1.0 (technoriku@gmail.com)"
)


wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
wiki_query.invoke("Who is Shubman Gill?")

"Page: Shubman Gill\nSummary: Shubman Gill (born 8 September 1999) is an Indian international cricketer who plays for the India national team. Gill captains India in Tests and ODIs, the Gujarat Titans in the Indian Premier League and Punjab when he plays first-class cricket. He has also captained India in T20I. A right-handed batsman, Gill represents Punjab in domestic cricket.\nIn ODI cricket, he is the fastest player to reach 2000 runs, in 38 innings, and 2500 runs, in 50 innings. He also holds the record for the youngest cricketer to score a double century in ODIs at the age of 23. With his country, he won the 2025 ICC Champions Trophy as vice captain. He made his List-A debut against Vidarbha in 2017 and first-class debut for Punjab against Bengal in the 2017–18 Ranji Trophy, scoring a half-century in the game, and 129 runs in the last match against Services.\nAs vice-captain of the Indian under-19 team, Gill scored 372 runs at an average of 124.00 in the 2018 Under-19 Cricket Worl

### **Custom Tool**

In [5]:
from langchain.tools import tool

@tool
def personal_info(name: str):
    """Use this tool to get personal information about someone."""

    info = {
        "Manabendu" : "He is a bengali fat boy!!",
        "Pramith" : "He is a cunning short mallu!!",
        "Jevial" : "The bosss baby of our gang!!"
    }

    return info.get(name, f"No information is available about the person {name}")

In [6]:
@tool
def wiki_tool(topic:str):
    """Use this tool for wikipedia search"""
    return wiki_query.invoke(topic)

In [7]:
@tool
def duckduckgo_tool(topic:str):
    """Use this tool for browser search and to get the latest updated about the given topic."""
    return duck_search.invoke(topic)

### **Tool Binding**

In [8]:
tools = [duckduckgo_tool, wiki_tool, personal_info]

model_with_tools = model.bind_tools(tools)

In [9]:
response = model_with_tools.invoke(
    "What is the latest scorecard update on ind vs sl test series"
)


In [ ]:
response.tool_calls

[{'name': 'duckduckgo_tool',
  'args': {'topic': 'India vs Sri Lanka test series latest scorecard update'},
  'id': 'call_4855905',
  'type': 'tool_call'}]

### **LangGraph Creation**

In [11]:
from typing import TypedDict, List

class graph_schema(TypedDict):
    messages: List

#### **Creating a LLM node**

In [ ]:
def llm_node(state: graph_schema) -> graph_schema:
    messages = state['messages']

    prompt = ChatMessagePromptTemplate.from